In [1]:

# CELL 1 — IMPORTS AND SETUP
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

from scipy.signal import butter, filtfilt, find_peaks
from scipy.stats import skew, kurtosis
from scipy.interpolate import CubicSpline
import pywt
import neurokit2 as nk
from tqdm import tqdm

# Add src to path
COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
sys.path.insert(0, os.path.join(COMPONENT_PATH, "src"))

from utils.config import (
    WESAD_PATH,
    WESAD_SUBJECT_IDS,
    WESAD_CHEST_FS,
    WINDOW_SEC,
    STEP_SEC,
    BOUNDARY_EXCLUSION_SEC,
    RESULTS_PATH
)

print("=" * 55)
print("PREPROCESSING PIPELINE — SETUP COMPLETE")
print("=" * 55)
print(f"WESAD path:          {WESAD_PATH}")
print(f"Subjects to process: {WESAD_SUBJECT_IDS}")
print(f"Sampling rate:       {WESAD_CHEST_FS} Hz")
print(f"Window size:         {WINDOW_SEC} seconds")
print(f"Step size:           {STEP_SEC} seconds")
print(f"Boundary exclusion:  {BOUNDARY_EXCLUSION_SEC} seconds")
print()
print("Imports successful — ready to build pipeline")

Config loaded successfully
PREPROCESSING PIPELINE — SETUP COMPLETE
WESAD path:          C:\Users\weha1\datasets\WESAD
Subjects to process: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
Sampling rate:       700 Hz
Window size:         60 seconds
Step size:           30 seconds
Boundary exclusion:  30 seconds

Imports successful — ready to build pipeline


In [2]:
# ============================================================
# CELL 2 — ECG PREPROCESSING 
#
# Changes from previous version:
#   1. Added adaptive fallback peak detection (6 tries)
#      → Fixes S9 (135 peaks) and S10 (296 peaks) failures
#      → S9/S10 had valid signals but unusual morphology
#      → Original Pan-Tompkins missed peaks due to amplitude
#
#   2. Added minimum expected peaks check
#      → Validates detection success before accepting result
#      → Minimum = recording duration × 40 bpm
#
#   3. Kept all previous fixes:
#      → Median-based contextual outlier removal
#      → Linear interpolation (not cubic spline)
#      → Hard limit 300-2000ms
#
# Validation results:
#   HR rises:    15/15 ✓
#   RMSSD drops: 10/15 ✓
#   pNN20 drops: 12/15 ✓
#   Atypical: S6, S7, S9, S14, S15 (Paper 12 individual variation)
#
# Papers: 10, 12 (bandpass 0.5-5Hz)
#         2, 10  (IBI validation 300-2000ms)
#         9      (pNN20 more sensitive than pNN50)
#         12     (4 physiological response groups)
# ============================================================

from scipy.signal import butter, filtfilt, find_peaks
from scipy.ndimage import median_filter

def preprocess_ecg(ecg_signal, fs=700):
    """
    Clean raw ECG and extract validated IBI sequence.
    Includes adaptive fallback for difficult signal morphologies.

    Parameters:
        ecg_signal : raw ECG array (mV)
        fs         : sampling rate (700 Hz for WESAD chest)

    Returns:
        ibi_clean  : cleaned IBI array in milliseconds
        r_peaks    : sample indices of detected R peaks
        ibi_times  : timestamp of each IBI in seconds
    """

    # ── Step 1: Flatten ──────────────────────────────────────
    ecg = ecg_signal.flatten().astype(np.float64)

    # ── Step 2: Bandpass filter 0.5 to 5 Hz ─────────────────
    # 0.5 Hz removes breathing drift
    # 5.0 Hz removes electrical noise
    # Papers 10 and 12 confirmed these values
    nyquist      = fs / 2
    b, a         = butter(2, [0.5/nyquist, 5.0/nyquist],
                          btype='bandpass')
    ecg_filtered = filtfilt(b, a, ecg)

    # ── Step 3: DC removal ───────────────────────────────────
    # Centers signal around zero for reliable peak detection
    win       = int(2 * fs)
    ma        = np.convolve(ecg_filtered,
                            np.ones(win)/win, mode='same')
    ecg_clean = ecg_filtered - ma

    # ── Step 4: Adaptive R-peak detection ────────────────────
    #
    # Problem discovered: S9 and S10 had unusual ECG morphology
    # that caused Pan-Tompkins to find only 135 and 296 peaks
    # instead of expected 5000-6000 peaks.
    #
    # Solution: 6 tries with different strategies
    # Stop as soon as one finds enough peaks
    #
    # "Enough peaks" = recording_minutes × 40 bpm minimum
    # This prevents accepting a failed detection
    #
    # Try 1: NeuroKit2 Pan-Tompkins (best for typical ECG)
    # Try 2: scipy 75th percentile threshold (standard)
    # Try 3: scipy 60th percentile (lower amplitude signals)
    # Try 4: scipy 50th percentile (very low amplitude)
    # Try 5: Inverted signal (some ECGs have flipped polarity)
    # Try 6: NeuroKit2 on inverted signal (last resort)

    min_distance = int(0.3 * fs)
    expected_min = max(10, int(len(ecg) / fs / 60 * 40))
    r_peaks      = None

    # Try 1: NeuroKit2 Pan-Tompkins
    try:
        signals, info = nk.ecg_process(
            ecg_clean, sampling_rate=fs
        )
        candidate = info['ECG_R_Peaks']
        if len(candidate) >= expected_min:
            r_peaks = candidate
    except Exception:
        pass

    # Try 2: scipy 75th percentile
    if r_peaks is None or len(r_peaks) < expected_min:
        candidate, _ = find_peaks(
            ecg_clean,
            distance=min_distance,
            height=np.percentile(ecg_clean, 75)
        )
        if len(candidate) >= expected_min:
            r_peaks = candidate

    # Try 3: scipy 60th percentile
    if r_peaks is None or len(r_peaks) < expected_min:
        candidate, _ = find_peaks(
            ecg_clean,
            distance=min_distance,
            height=np.percentile(ecg_clean, 60)
        )
        if len(candidate) >= expected_min:
            r_peaks = candidate

    # Try 4: scipy 50th percentile
    if r_peaks is None or len(r_peaks) < expected_min:
        candidate, _ = find_peaks(
            ecg_clean,
            distance=min_distance,
            height=np.percentile(ecg_clean, 50)
        )
        if len(candidate) >= expected_min:
            r_peaks = candidate

    # Try 5: Inverted signal scipy
    # Some ECGs have flipped polarity — R peak is a trough
    if r_peaks is None or len(r_peaks) < expected_min:
        ecg_inv       = -ecg_clean
        candidate, _  = find_peaks(
            ecg_inv,
            distance=min_distance,
            height=np.percentile(ecg_inv, 75)
        )
        if len(candidate) >= expected_min:
            r_peaks = candidate

    # Try 6: NeuroKit2 on inverted signal
    if r_peaks is None or len(r_peaks) < expected_min:
        try:
            ecg_inv       = -ecg_clean
            signals, info = nk.ecg_process(
                ecg_inv, sampling_rate=fs
            )
            candidate = info['ECG_R_Peaks']
            if len(candidate) >= expected_min:
                r_peaks = candidate
        except Exception:
            pass

    if r_peaks is None or len(r_peaks) < 5:
        return None, None, None

    # ── Step 5: Calculate IBI ────────────────────────────────
    ibi_raw   = np.diff(r_peaks) / fs * 1000
    ibi_times = r_peaks[:-1] / fs

    # ── Step 6: Hard physiological limit ─────────────────────
    # 300ms = 200 bpm maximum, 2000ms = 30 bpm minimum
    # Papers 2 and 10 confirmed these boundaries
    valid_hard = (ibi_raw >= 300) & (ibi_raw <= 2000)

    # ── Step 7: Median-based contextual outlier removal ──────
    # Removes IBIs that deviate >20% from local 5-beat median
    # Example: IBI=1800ms surrounded by 800ms IBIs → removed
    if valid_hard.sum() > 10:
        ibi_temp              = ibi_raw.copy()
        ibi_temp[~valid_hard] = np.median(ibi_raw[valid_hard])
        local_median          = median_filter(ibi_temp, size=5)
        deviation             = np.abs(ibi_raw - local_median) \
                                / local_median
        valid_median          = deviation < 0.20
        valid_final           = valid_hard & valid_median
    else:
        valid_final = valid_hard

    # ── Step 8: Linear interpolation ─────────────────────────
    # Fills rejected IBIs smoothly
    # Linear is safer than cubic spline for physiological data
    if valid_final.sum() < 5:
        return None, None, None

    x_valid   = np.where(valid_final)[0]
    x_all     = np.arange(len(ibi_raw))
    ibi_clean = np.interp(x_all, x_valid, ibi_raw[valid_final])

    # ── Step 9: Final hard clip ───────────────────────────────
    ibi_clean = np.clip(ibi_clean, 300, 2000)

    return ibi_clean, r_peaks, ibi_times


# ── Validation on all 15 subjects ────────────────────────────
print("=" * 65)
print("ECG PIPELINE — FINAL VALIDATION WITH ADAPTIVE DETECTION")
print("=" * 65)
print(f"{'Subj':6s} {'Base HR':9s} {'Stress HR':11s} "
      f"{'RMSSD B':9s} {'RMSSD S':9s} "
      f"{'Peaks':8s} {'Status':10s}")
print("-" * 65)

SUBJECT_IDS   = [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
final_results = []
atypical      = []

for sid in SUBJECT_IDS:
    try:
        pkl_path = os.path.join(
            WESAD_PATH, f"S{sid}", f"S{sid}.pkl"
        )
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f, encoding='latin1')

        ecg_full   = data['signal']['chest']['ECG'].flatten()
        raw_labels = data['label'].flatten()

        # Process full signal to get peak count
        ibi_full, peaks_full, times_full = preprocess_ecg(
            ecg_full, fs=700
        )

        base_idx   = np.where(raw_labels == 1)[0]
        stress_idx = np.where(raw_labels == 2)[0]

        # Get IBI for baseline and stress periods
        base_sec   = base_idx[0]   / 700
        base_end   = base_idx[-1]  / 700
        stress_sec = stress_idx[0] / 700
        stress_end = stress_idx[-1]/ 700

        ibi_b = ibi_full[
            (times_full >= base_sec) &
            (times_full <  base_end)
        ]
        ibi_s = ibi_full[
            (times_full >= stress_sec) &
            (times_full <  stress_end)
        ]

        if len(ibi_b) < 5 or len(ibi_s) < 5:
            print(f"S{sid:2d}    INSUFFICIENT IBI")
            continue

        hr_b    = 60000 / np.mean(ibi_b)
        hr_s    = 60000 / np.mean(ibi_s)
        rmssd_b = np.sqrt(np.mean(np.diff(ibi_b)**2))
        rmssd_s = np.sqrt(np.mean(np.diff(ibi_s)**2))
        pnn20_b = np.mean(np.abs(np.diff(ibi_b)) > 20) * 100
        pnn20_s = np.mean(np.abs(np.diff(ibi_s)) > 20) * 100
        sdnn_b  = np.std(ibi_b)
        sdnn_s  = np.std(ibi_s)

        hr_ok    = hr_s    > hr_b
        rmssd_ok = rmssd_s < rmssd_b

        if hr_ok and rmssd_ok:
            status = "TYPICAL  "
        elif hr_ok and not rmssd_ok:
            status = "ATYPICAL "
            atypical.append(sid)
        else:
            status = "CHECK    "

        n_peaks = len(peaks_full) if peaks_full is not None else 0

        print(f"S{sid:2d}    "
              f"{hr_b:7.1f}   "
              f"{hr_s:9.1f}   "
              f"{rmssd_b:7.1f}   "
              f"{rmssd_s:7.1f}   "
              f"{n_peaks:7d}   "
              f"{status}")

        final_results.append({
            'sid':     sid,
            'hr_b':    hr_b,    'hr_s':    hr_s,
            'rmssd_b': rmssd_b, 'rmssd_s': rmssd_s,
            'pnn20_b': pnn20_b, 'pnn20_s': pnn20_s,
            'sdnn_b':  sdnn_b,  'sdnn_s':  sdnn_s,
            'n_peaks': n_peaks
        })

    except Exception as e:
        print(f"S{sid:2d}    ERROR: {e}")

print("-" * 65)

if final_results:
    n        = len(final_results)
    hr_ok    = sum(1 for r in final_results if r['hr_s']    > r['hr_b'])
    rmssd_ok = sum(1 for r in final_results if r['rmssd_s'] < r['rmssd_b'])
    pnn20_ok = sum(1 for r in final_results if r['pnn20_s'] < r['pnn20_b'])
    sdnn_ok  = sum(1 for r in final_results if r['sdnn_s']  < r['sdnn_b'])

    avg = lambda key: np.mean([r[key] for r in final_results])

    print(f"\nFINAL VALIDATION SUMMARY:")
    print(f"  HR rises    (15/15 expected): {hr_ok}/{n}")
    print(f"  RMSSD drops (most expected):  {rmssd_ok}/{n}")
    print(f"  pNN20 drops (most expected):  {pnn20_ok}/{n}")
    print(f"  SDNN drops  (unreliable):     {sdnn_ok}/{n}")

    print(f"\nPEAK COUNTS (should all be 4000+ for full signal):")
    for r in final_results:
        flag = "✓" if r['n_peaks'] > 3000 else "✗ LOW"
        print(f"  S{r['sid']:2d}: {r['n_peaks']:6d} peaks  {flag}")

    print(f"\nMEAN VALUES:")
    print(f"  Baseline RMSSD: {avg('rmssd_b'):.1f}ms  "
          f"Stress RMSSD: {avg('rmssd_s'):.1f}ms")
    print(f"  Baseline HR:    {avg('hr_b'):.1f}bpm  "
          f"Stress HR:    {avg('hr_s'):.1f}bpm")

    print(f"\nATYPICAL RESPONDERS:")
    if atypical:
        print(f"  Subjects: {atypical}")
        print(f"  Genuine individual variation — Paper 12")
        print(f"  LOSO handles naturally")

    print(f"\nECG PIPELINE STATUS:")
    low_peaks = [r for r in final_results if r['n_peaks'] < 3000]
    if len(low_peaks) == 0 and hr_ok >= 14:
        print(f"  VALIDATED — S9 and S10 fixed")
        print(f"  Ready to rerun Cell 8 full pipeline")
    elif len(low_peaks) > 0:
        print(f"  Still low peaks for: "
              f"{[r['sid'] for r in low_peaks]}")
        print(f"  Paste output for further investigation")
    else:
        print(f"  VALIDATED — proceed to Cell 8")

ECG PIPELINE — FINAL VALIDATION WITH ADAPTIVE DETECTION
Subj   Base HR   Stress HR   RMSSD B   RMSSD S   Peaks    Status    
-----------------------------------------------------------------
S 2       72.1        76.6      56.7      44.7      6898   TYPICAL  
S 3       54.1        86.6     107.4      54.7      6408   TYPICAL  
S 4       60.6        79.6      82.2      42.4      6956   TYPICAL  
S 5       62.8        89.2      30.0      37.4      7246   ATYPICAL 
S 6       69.2        78.6      37.0      49.7      8367   ATYPICAL 
S 7       70.6        83.1      54.1      78.9      6118   ATYPICAL 
S 8       70.7        90.6      51.7      35.2      6878   TYPICAL  
S 9       77.2        81.4      41.2      45.3      7153   ATYPICAL 
S10       98.9       107.8      27.6      31.4      9101   ATYPICAL 
S11       74.1       123.3      48.4      31.8      7838   TYPICAL  
S13       86.9       108.2      27.7      22.1      8040   TYPICAL  
S14       70.3       111.9      42.0      64.7    

In [4]:
# ============================================================
# CELL 3 — EDA PREPROCESSING
# Converts raw EDA signal into tonic/phasic components
#
# What this does:
#   Raw EDA → zero/near-zero replacement → linear-drift removal
#   (fixes S2/S8 electrode drift) → IQR spike removal →
#   low-pass filter → SCL/SCR separation → wavelet
#   decomposition → signal derivatives
#
# Consolidated from an earlier version that lacked drift
# correction (used only 'raw EDA -> IQR -> filter', no detrend)
# once diagnosis showed S2/S8 had downward electrode drift —
# same function used everywhere in this notebook now, including
# the full 15-subject pipeline (Cell 8). discard_minutes dropped
# from the signature — never actually passed non-zero anywhere.
#
# Papers: 5  (filter specs: 5Hz, 0.05Hz)
#         7  (frequency boundaries: 0.16Hz)
#         12 (IQR removal, peak width)
#         16 (wavelet, derivatives, first 15min discard)
# ============================================================

import pickle
import os

# Load S2 for testing
test_pkl  = os.path.join(WESAD_PATH, "S2", "S2.pkl")
with open(test_pkl, 'rb') as f:
    test_data = pickle.load(f, encoding='latin1')

from scipy.signal import detrend as scipy_detrend

def preprocess_eda(eda_signal, fs=700):
    """
    Final EDA preprocessing with drift correction.
    
    Key insight from diagnostic:
      Values ARE already in microsiemens
      S2 and S8 have downward electrode drift
      Solution: remove linear trend before processing
    
    Papers: 5  (filter specs)
            7  (frequency boundaries)
            12 (IQR removal)
            16 (wavelet, derivatives)
    """

    eda = eda_signal.flatten().astype(np.float64)

    # ── Step 1: Replace zeros and near-zeros ─────────────────
    valid_mask = eda > 0.01
    if valid_mask.sum() > 100:
        valid_median     = np.median(eda[valid_mask])
        eda[~valid_mask] = valid_median

    # ── Step 2: Remove linear drift ──────────────────────────
    # This fixes S2 and S8 electrode drift issue
    # Removes slow linear trend while preserving
    # stress-related changes which are faster
    # Standard technique in EDA literature
    eda_detrended = scipy_detrend(eda, type='linear')

    # Shift back to positive range
    # (detrend centers around zero)
    eda_detrended = eda_detrended - eda_detrended.min() + 0.1

    # ── Step 3: IQR outlier removal ──────────────────────────
    Q1  = np.percentile(eda_detrended, 25)
    Q3  = np.percentile(eda_detrended, 75)
    IQR = Q3 - Q1

    lower        = Q1 - 1.5 * IQR
    upper        = Q3 + 1.5 * IQR
    outlier_mask = (eda_detrended < lower) | \
                   (eda_detrended > upper)

    if outlier_mask.sum() > 0:
        x_all   = np.arange(len(eda_detrended))
        x_clean = x_all[~outlier_mask]
        y_clean = eda_detrended[~outlier_mask]
        if len(x_clean) > 2:
            eda_detrended[outlier_mask] = np.interp(
                x_all[outlier_mask], x_clean, y_clean
            )

    # ── Step 4: Low-pass Butterworth 5 Hz ────────────────────
    nyquist      = fs / 2
    b_low, a_low = butter(2, 5.0/nyquist, btype='lowpass')
    eda_filtered = filtfilt(b_low, a_low, eda_detrended)

    # ── Step 5: SCL/SCR separation ───────────────────────────
    b_high, a_high = butter(
        2, 0.05/nyquist, btype='highpass'
    )
    eda_phasic = filtfilt(b_high, a_high, eda_filtered)
    eda_tonic  = eda_filtered - eda_phasic

    # ── Step 6: Derivatives ──────────────────────────────────
    deriv1 = np.gradient(eda_filtered)
    deriv2 = np.gradient(deriv1)

    # ── Step 7: Haar Wavelet 5-level ─────────────────────────
    try:
        wavelet_coeffs = pywt.wavedec(
            eda_filtered, 'haar', level=5
        )
    except Exception:
        wavelet_coeffs = pywt.wavedec(
            eda_filtered, 'haar', level=3
        )

    return {
        'raw':            eda,
        'filtered':       eda_filtered,
        'tonic':          eda_tonic,
        'phasic':         eda_phasic,
        'deriv1':         deriv1,
        'deriv2':         deriv2,
        'wavelet_coeffs': wavelet_coeffs
    }


# ── Test on Subject S2 ──────────────────────────────────────
print("Testing EDA preprocessing on S2...")

eda_signal = test_data['signal']['chest']['EDA']
eda_5min   = eda_signal[:210000]

eda_components = preprocess_eda(eda_5min, fs=700)

print(f"\nEDA preprocessing results (first 5 minutes of S2):")
print(f"  Raw EDA range:      "
      f"{eda_components['raw'].min():.3f} to "
      f"{eda_components['raw'].max():.3f} uS")
print(f"  Tonic mean:         {eda_components['tonic'].mean():.3f} uS")
print(f"  Phasic std:         {eda_components['phasic'].std():.4f} uS")
print(f"  Deriv1 mean abs:    {np.abs(eda_components['deriv1']).mean():.6f}")
print(f"  Wavelet levels:     {len(eda_components['wavelet_coeffs'])}")

print(f"\nSanity check (Paper 10 reference — baseline):")
print(f"  Expected tonic ~4.2 uS:  got {eda_components['tonic'].mean():.2f} uS")

print("\nEDA preprocessing function verified")


print()
print("=" * 65)
print("EDA VALIDATION — BASELINE vs STRESS ALL SUBJECTS")
print("=" * 65)
print("Expected: SCL_mean RISES from baseline to stress")
print("Paper 10: Normal=4.2 uS → Stress=14.7 uS")
print("=" * 65)
print(f"{'Subj':6s} {'Base SCL':10s} {'Stress SCL':12s} "
      f"{'Change':10s} {'Direction':12s}")
print("-" * 65)

eda_results = []
atypical_eda = []

for sid in SUBJECT_IDS:
    try:
        pkl_path = os.path.join(
            WESAD_PATH, f"S{sid}", f"S{sid}.pkl"
        )
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f, encoding='latin1')

        eda_full   = data['signal']['chest']['EDA'].flatten()
        raw_labels = data['label'].flatten()

        # Baseline EDA
        base_idx = np.where(raw_labels == 1)[0]
        eda_base = eda_full[base_idx[0]:base_idx[-1]]

        # Stress EDA
        stress_idx = np.where(raw_labels == 2)[0]
        eda_stress = eda_full[stress_idx[0]:stress_idx[-1]]

        # Preprocess both
        comp_b = preprocess_eda(eda_base,   fs=700)
        comp_s = preprocess_eda(eda_stress, fs=700)

        scl_b  = np.mean(comp_b['tonic'])
        scl_s  = np.mean(comp_s['tonic'])
        change = scl_s - scl_b

        direction = "↑ correct   " if change > 0 \
                    else "↓ unexpected"

        if change <= 0:
            atypical_eda.append(sid)

        print(f"S{sid:2d}      "
              f"{scl_b:8.3f}   "
              f"{scl_s:10.3f}   "
              f"{change:+8.3f}   "
              f"{direction}")

        eda_results.append({
            'sid': sid,
            'scl_b': scl_b,
            'scl_s': scl_s,
            'change': change
        })

    except Exception as e:
        print(f"S{sid:2d}     ERROR: {e}")

print("-" * 65)

if eda_results:
    correct = sum(1 for r in eda_results if r['change'] > 0)
    n       = len(eda_results)
    avg_b   = np.mean([r['scl_b'] for r in eda_results])
    avg_s   = np.mean([r['scl_s'] for r in eda_results])

    print(f"\nSUMMARY:")
    print(f"  SCL rises correctly: {correct}/{n}")
    print(f"\nMEAN SCL VALUES:")
    print(f"  Baseline SCL: {avg_b:.3f} uS  "
          f"(Paper 10 expected ~4.2 uS)")
    print(f"  Stress SCL:   {avg_s:.3f} uS  "
          f"(Paper 10 expected ~14.7 uS)")
    print(f"  Mean change:  {avg_s-avg_b:+.3f} uS  "
          f"({'↑ correct' if avg_s > avg_b else '↓ wrong'})")

    if atypical_eda:
        print(f"\nATYPICAL EDA RESPONDERS: {atypical_eda}")
        print(f"  SCL drops during stress — individual variation")

    if correct >= 10:
        print(f"\nEDA PIPELINE STATUS: VALIDATED")
        print(f"Ready to proceed to Cell 4 — ACC motion gating")
    else:
        print(f"\nEDA PIPELINE STATUS: Needs investigation")

Testing EDA preprocessing on S2...

EDA preprocessing results (first 5 minutes of S2):
  Raw EDA range:      4.918 to 7.577 uS
  Tonic mean:         0.898 uS
  Phasic std:         0.1709 uS
  Deriv1 mean abs:    0.000132
  Wavelet levels:     6

Sanity check (Paper 10 reference — baseline):
  Expected tonic ~4.2 uS:  got 0.90 uS

EDA preprocessing function verified

EDA VALIDATION — BASELINE vs STRESS ALL SUBJECTS
Expected: SCL_mean RISES from baseline to stress
Paper 10: Normal=4.2 uS → Stress=14.7 uS
Subj   Base SCL   Stress SCL   Change     Direction   
-----------------------------------------------------------------
S 2         0.835        0.647     -0.189   ↓ unexpected
S 3         0.930        1.783     +0.853   ↑ correct   
S 4         0.475        0.960     +0.485   ↑ correct   
S 5         0.733        1.164     +0.431   ↑ correct   
S 6         0.869        1.107     +0.239   ↑ correct   
S 7         0.564        0.714     +0.150   ↑ correct   
S 8         0.365        0.39

In [5]:
# ============================================================
# CELL 4 — ACC MOTION GATING
# Uses accelerometer for two purposes:
#   1. Detect and reject motion-contaminated windows
#   2. Keep ACC magnitude as predictive stress feature
#
# Papers: 1, 4, 5, 8 (motion gating necessity)
#         11 (ACC mean is top SHAP stress feature — dual role)
# ============================================================

def process_acc(acc_signal, fs=700, threshold=1.2):
    """
    Process accelerometer for motion gating and feature extraction.

    Parameters:
        acc_signal : raw ACC array shape (N, 3) — x, y, z axes
        fs         : sampling rate (700 Hz)
        threshold  : motion threshold in g (default 1.2g)

    Returns:
        magnitude   : resultant acceleration per sample
        clean_mask  : True = clean, False = motion contaminated
        motion_pct  : percentage of motion-contaminated samples
    """

    # ── Step 1: Ensure correct shape ────────────────────────
    acc = np.array(acc_signal)
    if acc.ndim == 1:
        acc = acc.reshape(-1, 1)

    # ── Step 2: Calculate resultant magnitude ───────────────
    #
    # WHY: Each axis measures one direction of movement
    #      You need the combined movement in all directions
    #
    # Formula: magnitude = sqrt(ax² + ay² + az²)
    #
    # When completely still:
    #   Only gravity acts on sensor = 1.0g downward
    #   magnitude ≈ 1.0g
    #
    # When moving arm:
    #   Additional acceleration in multiple axes
    #   magnitude > 1.2g
    #
    # Example:
    #   Still:   ax=0.01, ay=0.02, az=1.00 → magnitude=1.001g
    #   Moving:  ax=0.45, ay=0.62, az=0.95 → magnitude=1.264g
    if acc.shape[1] >= 3:
        magnitude = np.sqrt(
            acc[:, 0]**2 +
            acc[:, 1]**2 +
            acc[:, 2]**2
        )
    else:
        magnitude = np.abs(acc[:, 0])

    # ── Step 3: Smooth with 1-second window ─────────────────
    #
    # WHY: Brief vibrations (cough, small movement) should not
    # reject an entire window. Smoothing averages over 1 second
    # so only sustained motion triggers rejection.
    #
    # Example:
    #   Raw magnitude:    [1.0, 1.0, 2.8, 1.0, 1.0, 1.0]
    #   After smoothing:  [1.1, 1.2, 1.3, 1.3, 1.1, 1.0]
    #   Still below 1.2 → brief cough does not reject window
    window_samples  = fs  # 1 second
    magnitude_smooth = np.convolve(
        magnitude,
        np.ones(window_samples) / window_samples,
        mode='same'
    )

    # ── Step 4: Create clean mask ───────────────────────────
    #
    # True  = clean window — use for feature extraction
    # False = motion contaminated — reject this window
    #
    # Threshold 1.2g chosen because:
    #   Gravity alone = 1.0g
    #   Small breathing movements add ~0.1-0.15g
    #   1.2g safely above normal breathing noise
    #   Papers 5, 8 confirmed this threshold range
    clean_mask   = magnitude_smooth <= threshold
    motion_pct   = (~clean_mask).mean() * 100

    return magnitude_smooth, clean_mask, motion_pct


# ── Test on all 15 subjects ──────────────────────────────────
print("=" * 65)
print("ACC MOTION GATING VALIDATION — ALL SUBJECTS")
print("=" * 65)
print("Expected:")
print("  Baseline: low motion (person sitting quietly)")
print("  Stress:   slightly more motion (speaking, gesturing)")
print("=" * 65)
print(f"{'Subj':6s} {'Base Motion%':14s} {'Stress Motion%':16s} "
      f"{'Base ACC':10s} {'Stress ACC':12s} {'Status':10s}")
print("-" * 65)

acc_results = []

for sid in SUBJECT_IDS:
    try:
        pkl_path = os.path.join(
            WESAD_PATH, f"S{sid}", f"S{sid}.pkl"
        )
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f, encoding='latin1')

        acc_full   = data['signal']['chest']['ACC']
        raw_labels = data['label'].flatten()

        # Baseline ACC
        base_idx = np.where(raw_labels == 1)[0]
        acc_base = acc_full[base_idx[0]:base_idx[-1]]

        # Stress ACC
        stress_idx = np.where(raw_labels == 2)[0]
        acc_stress = acc_full[stress_idx[0]:stress_idx[-1]]

        # Process both
        mag_b, mask_b, motion_b = process_acc(acc_base,   fs=700)
        mag_s, mask_s, motion_s = process_acc(acc_stress, fs=700)

        acc_mean_b = np.mean(mag_b)
        acc_mean_s = np.mean(mag_s)

        # Status check
        # Baseline should have lower motion than stress
        # (person is sitting vs standing/speaking during TSST)
        if motion_b < motion_s:
            status = "expected "
        else:
            status = "check    "

        print(f"S{sid:2d}      "
              f"{motion_b:10.1f}%     "
              f"{motion_s:12.1f}%     "
              f"{acc_mean_b:8.3f}   "
              f"{acc_mean_s:10.3f}   "
              f"{status}")

        acc_results.append({
            'sid':       sid,
            'motion_b':  motion_b,
            'motion_s':  motion_s,
            'acc_mean_b': acc_mean_b,
            'acc_mean_s': acc_mean_s
        })

    except Exception as e:
        print(f"S{sid:2d}     ERROR: {e}")

print("-" * 65)

if acc_results:
    n = len(acc_results)

    avg_motion_b  = np.mean([r['motion_b']  for r in acc_results])
    avg_motion_s  = np.mean([r['motion_s']  for r in acc_results])
    avg_acc_b     = np.mean([r['acc_mean_b'] for r in acc_results])
    avg_acc_s     = np.mean([r['acc_mean_s'] for r in acc_results])

    more_motion   = sum(
        1 for r in acc_results if r['motion_s'] > r['motion_b']
    )

    print(f"\nSUMMARY:")
    print(f"  More motion during stress: {more_motion}/{n}  "
          f"(expected — TSST involves standing and speaking)")
    print(f"\nMEAN VALUES:")
    print(f"  Baseline  motion:   {avg_motion_b:.1f}%  "
          f"ACC mean: {avg_acc_b:.3f}g")
    print(f"  Stress    motion:   {avg_motion_s:.1f}%  "
          f"ACC mean: {avg_acc_s:.3f}g")

    print(f"\nDUAL ROLE CONFIRMATION:")
    print(f"  Role 1 — Motion gating: windows with magnitude >1.2g rejected")
    print(f"  Role 2 — Predictive feature (Paper 11 SHAP):")
    print(f"    Baseline ACC mean: {avg_acc_b:.3f}g")
    print(f"    Stress ACC mean:   {avg_acc_s:.3f}g")
    if avg_acc_s != avg_acc_b:
        print(f"    Difference: {avg_acc_s - avg_acc_b:+.3f}g  "
              f"← confirms ACC captures stress-related movement")
    
    # Check how many windows would be rejected per subject
    print(f"\nWINDOW REJECTION ESTIMATE:")
    print(f"  Average motion contamination during baseline: "
          f"{avg_motion_b:.1f}%")
    print(f"  Average motion contamination during stress:   "
          f"{avg_motion_s:.1f}%")
    print(f"  Windows with >10% motion contamination = rejected")
    print(f"  This protects HRV and GSR features from artifacts")

    if avg_motion_b < 20:
        print(f"\nACC PIPELINE STATUS: VALIDATED")
        print(f"  Motion contamination is acceptably low")
        print(f"  Ready to proceed to Cell 5 — PSS label creation")
    else:
        print(f"\nACC PIPELINE STATUS: High motion — investigate")

ACC MOTION GATING VALIDATION — ALL SUBJECTS
Expected:
  Baseline: low motion (person sitting quietly)
  Stress:   slightly more motion (speaking, gesturing)
Subj   Base Motion%   Stress Motion%   Base ACC   Stress ACC   Status    
-----------------------------------------------------------------
S 2             0.0%              0.0%        0.950        0.929   check    
S 3             0.0%              0.2%        0.926        0.912   expected 
S 4             0.0%              0.0%        0.968        0.924   check    
S 5             0.0%              0.0%        0.927        0.917   check    
S 6             0.0%              0.0%        0.954        0.938   check    
S 7             0.0%              0.0%        0.921        0.922   check    
S 8             0.0%              0.0%        0.951        0.918   check    
S 9             0.0%              0.0%        0.924        0.932   check    
S10             0.0%              0.0%        0.915        0.931   check    
S11       

In [6]:
# ============================================================
# CELL 5 — BINARY LABEL CREATION AND VERIFICATION
# Converts WESAD original labels to binary (Normal/Stress) labels
#
# WESAD original -> binary labels:
#   0 (transient)   -> -1 (exclude)
#   1 (baseline)    -> 0  (Normal)
#   2 (stress)      -> 1  (Stress) -- entire TSST condition, no
#                     internal split.
#   3 (amusement)   -> -1 (exclude)
#   4 (meditation)  -> -1 (exclude)
#   boundaries +/-30s -> -1 (exclude)
#
# WESAD has no real ground truth for an intermediate "pre-stress"
# state. An earlier version of this pipeline carved the first 90s
# of TSST into a synthetic Pre-stress class (citing GSR latency /
# gradual-buildup literature) -- but training on it topped out at
# F1=0.286 regardless of method, a symptom of a label with no
# reliable ground truth (see physiological_signals/notebooks/
# 05_model_training.ipynb for the binary model this feeds, and the
# project's literature review for the full reasoning). Removed
# entirely at the source rather than created and merged away
# downstream.
# ============================================================

from scipy.ndimage import median_filter

def create_pss_labels(raw_labels, fs=700, boundary_sec=30):
    """
    Convert WESAD raw labels to binary (Normal=0/Stress=1) labels.

    Parameters:
        raw_labels   : original WESAD label array
        fs           : sampling rate (700 Hz)
        boundary_sec : seconds to exclude around real WESAD
                       condition transitions

    Returns:
        pss : array with values -1, 0, 1
    """

    labels = raw_labels.flatten().astype(int)
    pss    = np.full(len(labels), -1, dtype=int)

    # Normal = Baseline (WESAD label 1)
    pss[labels == 1] = 0

    # Stress = entire TSST condition (WESAD label 2) -- no internal split
    pss[labels == 2] = 1

    # Exclude +/-30s boundaries around any real WESAD condition change
    boundary_samples = int(boundary_sec * fs)

    for i in range(1, len(labels)):
        if labels[i] != labels[i - 1]:
            excl_start = max(0, i - boundary_samples)
            excl_end   = min(len(pss), i + boundary_samples)
            pss[excl_start:excl_end] = -1

    return pss


# ── Verify on all 15 subjects ────────────────────────────────
print("=" * 65)
print("BINARY LABEL CREATION -- VERIFICATION ALL SUBJECTS")
print("=" * 65)
print(f"{'Subj':6s} {'Normal':8s} {'Stress':8s} {'Excluded':10s}")
print("-" * 65)

label_results = []

for sid in SUBJECT_IDS:
    try:
        pkl_path = os.path.join(
            WESAD_PATH, f"S{sid}", f"S{sid}.pkl"
        )
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f, encoding='latin1')

        raw_labels = data['label'].flatten()
        pss        = create_pss_labels(raw_labels, fs=700)

        n_normal   = (pss == 0).sum()
        n_stress   = (pss == 1).sum()
        n_excluded = (pss == -1).sum()

        normal_min = n_normal / 700 / 60
        stress_min = n_stress / 700 / 60

        print(f"S{sid:2d}     "
              f"{n_normal:7d}  "
              f"{n_stress:7d}  "
              f"{n_excluded:9d}")

        label_results.append({
            'sid':        sid,
            'n_normal':   n_normal,
            'n_stress':   n_stress,
            'normal_min': normal_min,
            'stress_min': stress_min
        })

    except Exception as e:
        print(f"S{sid:2d}     ERROR: {e}")

print("-" * 65)

if label_results:
    n = len(label_results)

    avg_normal = np.mean([r['normal_min'] for r in label_results])
    avg_stress = np.mean([r['stress_min'] for r in label_results])

    total_normal = sum(r['n_normal'] for r in label_results)
    total_stress = sum(r['n_stress'] for r in label_results)
    total_all    = total_normal + total_stress

    print(f"\nSUMMARY:")
    print(f"  Avg Normal duration: {avg_normal:.1f} min per subject")
    print(f"  Avg Stress duration: {avg_stress:.1f} min per subject")

    print(f"\nTOTAL LABELLED SAMPLES ACROSS 15 SUBJECTS:")
    print(f"  Normal: {total_normal:8d} ({total_normal/total_all*100:.1f}%)")
    print(f"  Stress: {total_stress:8d} ({total_stress/total_all*100:.1f}%)")

    print(f"\nCLASS IMBALANCE:")
    ratio = total_normal / max(total_stress, 1)
    print(f"  Normal:Stress ratio = {ratio:.1f}:1")
    print(f"  class_weight='balanced' / sample_weight used downstream to compensate")

    print(f"\nESTIMATED WINDOWS (60s window, 30s step):")
    est_windows = int(
        (avg_normal + avg_stress) * 60 / 30 * n
    )
    print(f"  ~{est_windows} total windows across all subjects")
    print(f"  Normal windows: ~{int(total_normal/700/30)} windows")
    print(f"  Stress windows: ~{int(total_stress/700/30)} windows")

    print(f"\nBINARY LABEL CREATION STATUS: COMPLETE")
    print(f"Ready to proceed to Cell 6 -- Feature Extraction")

BINARY LABEL CREATION -- VERIFICATION ALL SUBJECTS
Subj   Normal   Stress   Excluded  
-----------------------------------------------------------------
S 2      758800   388500    3108000
S 3      756000   406000    3383100
S 4      768601   402500    3324999
S 5      796600   409500    3174500
S 6      784000   413000    3752700
S 7      788200   406000    2472400
S 8      776300   427000    2622900
S 9      784000   409500    2462600
S10      784000   465500    2597700
S11      784000   434000    2445100
S13      784001   422800    2669099
S14      784000   430500    2669100
S15      780500   438200    2457700
S16      784000   429101    2728599
S17      784700   464100    2895200
-----------------------------------------------------------------

SUMMARY:
  Avg Normal duration: 18.6 min per subject
  Avg Stress duration: 10.1 min per subject

TOTAL LABELLED SAMPLES ACROSS 15 SUBJECTS:
  Normal: 11697702 (64.8%)
  Stress:  6346201 (35.2%)

CLASS IMBALANCE:
  Normal:Stress ratio = 1.8

In [7]:
# ============================================================
# CELL 6 — FEATURE EXTRACTION
# Extracts all features from one 60-second window
#
# Input:  one window of IBI, EDA components, ACC magnitude
# Output: dictionary of ~36 features
#
# Feature groups:
#   HRV Time Domain:    SDNN, RMSSD, AVNN, pNN50, pNN20,
#                       MeanHR, SDHR
#   HRV Frequency:      LF, HF, LF_HF, VLF, LFnu, HFnu
#   HRV Nonlinear:      SD1, SD2
#   GSR Tonic:          SCL_mean, SCL_std, SCL_slope
#   GSR Phasic:         SCR_count, SCR_amp, SCR_width
#   GSR Statistical:    EDA_mean, EDA_std, EDA_slope,
#                       EDA_q10, EDA_q25, EDA_q75, EDA_q90
#   GSR Derivatives:    deriv1_mean, deriv1_std, deriv2_mean
#   GSR Wavelet:        wavelet_approx_mean, approx_energy,
#                       detail1_energy, detail2_energy,
#                       detail3_energy
#   ACC:                ACC_mean, ACC_std, ACC_range
#
# Papers: 2,3,9,10 (HRV features)
#         5,7,8,10,12 (GSR features)
#         12 (SCR peak width novel feature)
#         16 (derivatives, wavelets, quantiles)
#         11 (ACC mean — top SHAP feature)
# ============================================================

from scipy.signal import find_peaks

def extract_hrv_features(ibi_window_ms):
    """
    Extract HRV features from IBI sequence in one window.

    Parameters:
        ibi_window_ms : IBI values in ms for this window

    Returns:
        Dictionary of HRV features or None if insufficient data
    """

    ibi = np.array(ibi_window_ms, dtype=np.float64)

    if len(ibi) < 5:
        return None

    features = {}

    # ── Time Domain ──────────────────────────────────────────
    # Papers 2, 3, 10 — most studied stress HRV features

    features['SDNN']   = np.std(ibi)
    features['RMSSD']  = np.sqrt(np.mean(np.diff(ibi)**2))
    features['AVNN']   = np.mean(ibi)
    features['MeanHR'] = 60000 / features['AVNN']
    features['SDHR']   = np.std(60000 / ibi)

    nn_diffs = np.abs(np.diff(ibi))
    features['pNN50']  = np.mean(nn_diffs > 50) * 100
    features['pNN20']  = np.mean(nn_diffs > 20) * 100  # Paper 9

    # ── Frequency Domain ─────────────────────────────────────
    # Papers 1, 3, 9, 10
    # LF/HF ratio is the key stress indicator
    # Rises under stress (sympathetic dominance)
    try:
        hrv_indices = nk.hrv_frequency(
            {'RRI': ibi},
            sampling_rate=1000,
            show=False
        )
        features['LF']    = float(
            hrv_indices.get('HRV_LF',   [np.nan]).iloc[0]
        )
        features['HF']    = float(
            hrv_indices.get('HRV_HF',   [np.nan]).iloc[0]
        )
        features['LF_HF'] = float(
            hrv_indices.get('HRV_LFHF', [np.nan]).iloc[0]
        )
        features['VLF']   = float(
            hrv_indices.get('HRV_VLF',  [np.nan]).iloc[0]
        )
        features['LFnu']  = float(
            hrv_indices.get('HRV_LFn',  [np.nan]).iloc[0]
        )
        features['HFnu']  = float(
            hrv_indices.get('HRV_HFn',  [np.nan]).iloc[0]
        )
    except Exception:
        for key in ['LF', 'HF', 'LF_HF', 'VLF', 'LFnu', 'HFnu']:
            features[key] = np.nan

    # ── Nonlinear — Poincaré Plot ─────────────────────────────
    # Paper 12: SD1 and SD2 capture short and long term variability
    if len(ibi) >= 4:
        diff_ibi       = np.diff(ibi)
        features['SD1'] = np.std(diff_ibi) / np.sqrt(2)
        sd2_sq          = max(
            0,
            2 * np.std(ibi)**2 - features['SD1']**2
        )
        features['SD2'] = np.sqrt(sd2_sq)
    else:
        features['SD1'] = np.nan
        features['SD2'] = np.nan

    return features


def extract_eda_features(eda_components, fs=700):
    """
    Extract EDA/GSR features from one window.

    Parameters:
        eda_components : dictionary from preprocess_eda
        fs             : sampling rate

    Returns:
        Dictionary of EDA features
    """

    features  = {}
    filtered  = eda_components['filtered']
    tonic     = eda_components['tonic']
    phasic    = eda_components['phasic']
    deriv1    = eda_components['deriv1']
    deriv2    = eda_components['deriv2']
    wavelets  = eda_components['wavelet_coeffs']
    n_samples = len(filtered)

    # ── Tonic SCL Features ────────────────────────────────────
    # Papers 5, 7, 8, 10
    # SCL_mean is the primary stress indicator
    # SCL_slope captures the rising trend (stress-onset indicator)
    features['SCL_mean']  = np.mean(tonic)
    features['SCL_std']   = np.std(tonic)
    features['SCL_slope'] = np.polyfit(
        np.arange(n_samples), tonic, 1
    )[0]

    # ── Overall EDA Statistics ────────────────────────────────
    features['EDA_mean']  = np.mean(filtered)
    features['EDA_std']   = np.std(filtered)
    features['EDA_range'] = np.max(filtered) - np.min(filtered)
    features['EDA_slope'] = np.polyfit(
        np.arange(n_samples), filtered, 1
    )[0]

    # ── Phasic SCR Features ───────────────────────────────────
    # Papers 5, 7, 8, 12
    # SCR_count: frequency of stress responses per minute
    # SCR_peak_width: novel feature from Paper 12
    #   wider peaks = deeper/longer stress response
    min_distance = int(1.0 * fs)   # minimum 1 second between peaks
    min_height   = 0.005           # minimum 0.005 uS amplitude
    min_width    = int(0.5 * fs)   # minimum 0.5 second width

    peaks, properties = find_peaks(
        phasic,
        height=min_height,
        distance=min_distance,
        width=min_width
    )

    duration_min = n_samples / fs / 60
    features['SCR_count'] = len(peaks) / max(duration_min, 0.01)

    if len(peaks) > 0:
        features['SCR_amp_mean']   = float(np.mean(phasic[peaks]))
        widths = properties.get('widths', np.array([0.0]))
        features['SCR_width_mean'] = float(
            np.mean(widths) / fs
        )
    else:
        features['SCR_amp_mean']   = 0.0
        features['SCR_width_mean'] = 0.0

    # ── Quantile Features ─────────────────────────────────────
    # Paper 16: robust to individual phasic pattern differences
    # Not affected by occasional outlier peaks
    features['EDA_q10'] = float(np.percentile(filtered, 10))
    features['EDA_q25'] = float(np.percentile(filtered, 25))
    features['EDA_q75'] = float(np.percentile(filtered, 75))
    features['EDA_q90'] = float(np.percentile(filtered, 90))

    # ── Derivative Features ───────────────────────────────────
    # Paper 16: best single predictor of stress level transitions
    # deriv1_mean: average rate of change
    #   → high value = signal changing rapidly = stress onset
    # deriv1_std: variability in rate of change
    #   → captures irregular stress responses
    features['deriv1_mean'] = float(np.mean(np.abs(deriv1)))
    features['deriv1_std']  = float(np.std(deriv1))
    features['deriv2_mean'] = float(np.mean(np.abs(deriv2)))

    # ── Wavelet Features ──────────────────────────────────────
    # Papers 14, 16: multi-timescale stress decomposition
    # Approximation = tonic slow stress buildup
    # Detail coefficients = phasic fast responses
    if len(wavelets) > 0:
        approx = wavelets[0]
        features['wavelet_approx_mean']   = float(np.mean(approx))
        features['wavelet_approx_energy'] = float(np.sum(approx**2))

    for level in range(1, min(4, len(wavelets))):
        detail = wavelets[level]
        features[f'wavelet_d{level}_energy'] = float(
            np.sum(detail**2)
        )

    return features


def extract_acc_features(magnitude_window):
    """
    Extract ACC features from one window.
    Paper 11: ACC mean is top SHAP stress predictor.
    """

    features = {}
    features['ACC_mean']  = float(np.mean(magnitude_window))
    features['ACC_std']   = float(np.std(magnitude_window))
    features['ACC_range'] = float(
        np.max(magnitude_window) - np.min(magnitude_window)
    )
    return features


def extract_all_features(ibi_window,
                         eda_window_components,
                         acc_window_magnitude,
                         fs=700):
    """
    Combine all feature groups into one dictionary.
    """

    all_features = {}

    # HRV features from ECG/IBI
    hrv = extract_hrv_features(ibi_window)
    if hrv is None:
        return None
    all_features.update(hrv)

    # GSR features from EDA
    eda = extract_eda_features(eda_window_components, fs=fs)
    all_features.update(eda)

    # ACC features
    acc = extract_acc_features(acc_window_magnitude)
    all_features.update(acc)

    return all_features


# ── Test feature extraction on S2 baseline window ────────────
print("=" * 65)
print("CELL 6 — FEATURE EXTRACTION TEST")
print("=" * 65)
print("Testing on S2 baseline (first 60-second window)...")
print()

# Load S2
test_pkl = os.path.join(WESAD_PATH, "S2", "S2.pkl")
with open(test_pkl, 'rb') as f:
    test_data = pickle.load(f, encoding='latin1')

raw_labels = test_data['label'].flatten()
ecg_full   = test_data['signal']['chest']['ECG'].flatten()
eda_full   = test_data['signal']['chest']['EDA'].flatten()
acc_full   = test_data['signal']['chest']['ACC']

# Find baseline start
base_idx    = np.where(raw_labels == 1)[0]
base_start  = base_idx[0]

# Extract first 60-second window from baseline
window_samples = int(60 * 700)
w_start        = base_start
w_end          = base_start + window_samples

# Process ECG for this window
ecg_window  = ecg_full[w_start:w_end]
ibi_full_b, r_peaks_b, ibi_times_b = preprocess_ecg(
    ecg_full[base_idx[0]:base_idx[-1]], fs=700
)

# Get IBI values that fall in this window
w_start_sec = w_start / 700
w_end_sec   = w_end   / 700
ibi_mask    = (
    (ibi_times_b >= w_start_sec) &
    (ibi_times_b <  w_end_sec)
)
ibi_window  = ibi_full_b[ibi_mask]

# Process EDA for this window
eda_full_comp = preprocess_eda(
    eda_full[base_idx[0]:base_idx[-1]], fs=700
)
offset     = w_start - base_idx[0]
eda_window = {}
for key in ['filtered','tonic','phasic','deriv1','deriv2']:
    eda_window[key] = eda_full_comp[key][offset:offset+window_samples]
try:
    eda_window['wavelet_coeffs'] = pywt.wavedec(
        eda_window['filtered'], 'haar', level=5
    )
except Exception:
    eda_window['wavelet_coeffs'] = pywt.wavedec(
        eda_window['filtered'], 'haar', level=3
    )

# Process ACC for this window
from scipy.ndimage import median_filter as mf
acc_window = acc_full[w_start:w_end]
mag_w, _, _ = process_acc(acc_window, fs=700)

# Extract all features
features = extract_all_features(ibi_window, eda_window, mag_w)

print(f"Features extracted: {len(features)} total")
print()

# Print all features grouped
hrv_keys = ['SDNN','RMSSD','AVNN','MeanHR','SDHR',
            'pNN50','pNN20','LF','HF','LF_HF',
            'VLF','LFnu','HFnu','SD1','SD2']
eda_keys = ['SCL_mean','SCL_std','SCL_slope',
            'EDA_mean','EDA_std','EDA_range','EDA_slope',
            'SCR_count','SCR_amp_mean','SCR_width_mean',
            'EDA_q10','EDA_q25','EDA_q75','EDA_q90',
            'deriv1_mean','deriv1_std','deriv2_mean',
            'wavelet_approx_mean','wavelet_approx_energy',
            'wavelet_d1_energy','wavelet_d2_energy',
            'wavelet_d3_energy']
acc_keys = ['ACC_mean','ACC_std','ACC_range']

print("HRV FEATURES (from ECG):")
for k in hrv_keys:
    v = features.get(k, np.nan)
    print(f"  {k:15s}: {v:.4f}")

print()
print("GSR FEATURES (from EDA):")
for k in eda_keys:
    v = features.get(k, np.nan)
    print(f"  {k:25s}: {v:.6f}")

print()
print("ACC FEATURES:")
for k in acc_keys:
    v = features.get(k, np.nan)
    print(f"  {k:15s}: {v:.4f}")

# ── Validate against Paper 10 reference values ───────────────
print()
print("=" * 65)
print("VALIDATION AGAINST PAPER 10 REFERENCE VALUES")
print("(This is a BASELINE window — should match Normal values)")
print("=" * 65)
checks = [
    ('SDNN',    features.get('SDNN',0),    62.3, 'ms', 'drop under stress'),
    ('RMSSD',   features.get('RMSSD',0),   48.6, 'ms', 'drop under stress'),
    ('LF_HF',   features.get('LF_HF',0),    1.2, '',   'rise under stress'),
    ('SCL_mean',features.get('SCL_mean',0), 4.2, 'uS', 'rise under stress'),
    ('MeanHR',  features.get('MeanHR',0),  65.0, 'bpm','rise under stress'),
]
for name, val, ref, unit, direction in checks:
    diff = val - ref
    print(f"  {name:12s}: got {val:7.2f}{unit:3s}  "
          f"ref ~{ref:.1f}{unit:3s}  "
          f"diff {diff:+.2f}  "
          f"({direction})")

print()
print("Note: Individual variation is expected.")
print("What matters is the DIRECTION of change")
print("Normal → Stress")
print("not the absolute match to Paper 10 averages")

print()
print("CELL 6 COMPLETE — Feature extraction functions ready")
print("Proceed to Cell 7 — Windowing")

CELL 6 — FEATURE EXTRACTION TEST
Testing on S2 baseline (first 60-second window)...

Features extracted: 40 total

HRV FEATURES (from ECG):
  SDNN           : 72.9284
  RMSSD          : 53.8446
  AVNN           : 843.7626
  MeanHR         : 71.1101
  SDHR           : 6.2956
  pNN50          : 31.4286
  pNN20          : 72.8571
  LF             : 0.0522
  HF             : 0.0099
  LF_HF          : 5.2605
  VLF            : nan
  LFnu           : 0.8232
  HFnu           : 0.1565
  SD1            : 38.0703
  SD2            : 95.8528

GSR FEATURES (from EDA):
  SCL_mean                 : 2.099052
  SCL_std                  : 0.149321
  SCL_slope                : -0.000012
  EDA_mean                 : 2.098911
  EDA_std                  : 0.150703
  EDA_range                : 0.462102
  EDA_slope                : -0.000012
  SCR_count                : 8.000000
  SCR_amp_mean             : 0.017772
  SCR_width_mean           : 1.746650
  EDA_q10                  : 1.877838
  EDA_q25         

In [8]:
# ============================================================
# CELL 7 — WINDOWING
# preprocess_eda() (with drift correction) now lives in Cell 3 —
# this cell only defines the windowing function that consumes it.
# ============================================================

# ── Updated windowing function ────────────────────────────────
def extract_windows_and_features(
    ibi_ms, ibi_times,
    eda_components_full,
    magnitude_full,
    clean_mask,
    pss_labels,
    fs=700,
    window_sec=60,
    step_sec=30,
    min_clean_ratio=0.90,
    min_ibi_count=10
):
    window_samples = int(window_sec * fs)
    step_samples   = int(step_sec   * fs)
    total_samples  = len(pss_labels)

    features_list = []
    labels_list   = []
    rejected      = {
        'excluded_label': 0,
        'mixed_label':    0,
        'motion':         0,
        'short_ibi':      0,
        'feat_error':     0
    }

    for start in range(
        0, total_samples - window_samples, step_samples
    ):
        end = start + window_samples

        window_labels = pss_labels[start:end]
        valid_labels  = window_labels[window_labels >= 0]
        unique_labels = np.unique(valid_labels)

        if len(unique_labels) == 0:
            rejected['excluded_label'] += 1
            continue
        if len(unique_labels) > 1:
            rejected['mixed_label'] += 1
            continue

        window_label = int(unique_labels[0])

        if clean_mask[start:end].mean() < min_clean_ratio:
            rejected['motion'] += 1
            continue

        start_sec  = start / fs
        end_sec    = end   / fs
        ibi_mask_w = (
            (ibi_times >= start_sec) &
            (ibi_times <  end_sec)
        )
        ibi_window = ibi_ms[ibi_mask_w]

        if len(ibi_window) < min_ibi_count:
            rejected['short_ibi'] += 1
            continue

        eda_window = {}
        for key in ['filtered','tonic','phasic',
                    'deriv1','deriv2']:
            eda_window[key] = \
                eda_components_full[key][start:end]

        try:
            eda_window['wavelet_coeffs'] = pywt.wavedec(
                eda_window['filtered'], 'haar', level=5
            )
        except Exception:
            eda_window['wavelet_coeffs'] = pywt.wavedec(
                eda_window['filtered'], 'haar', level=3
            )

        acc_window = magnitude_full[start:end]

        try:
            all_feats = extract_all_features(
                ibi_window, eda_window,
                acc_window, fs=fs
            )
            if all_feats is None:
                rejected['feat_error'] += 1
                continue
            features_list.append(all_feats)
            labels_list.append(window_label)
        except Exception:
            rejected['feat_error'] += 1
            continue

    return features_list, labels_list, rejected


# ── Validate fix on all subjects ─────────────────────────────
print("=" * 65)
print("EDA DETREND FIX — VALIDATION ALL SUBJECTS")
print("=" * 65)
print("Expected: stress SCL > baseline SCL for all subjects")
print("=" * 65)
print(f"{'Subj':6s} {'Base SCL':10s} {'Stress SCL':12s} "
      f"{'Change':10s} {'Direction':12s}")
print("-" * 65)

SUBJECT_IDS = [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
eda_results = []

for sid in SUBJECT_IDS:
    pkl_path = os.path.join(
        WESAD_PATH, f"S{sid}", f"S{sid}.pkl"
    )
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    eda_full   = data['signal']['chest']['EDA'].flatten()
    raw_labels = data['label'].flatten()

    base_idx   = np.where(raw_labels == 1)[0]
    stress_idx = np.where(raw_labels == 2)[0]

    # Apply new preprocessing
    comp = preprocess_eda(eda_full, fs=700)

    scl_base   = np.mean(
        comp['tonic'][base_idx[0]:base_idx[-1]]
    )
    scl_stress = np.mean(
        comp['tonic'][stress_idx[0]:stress_idx[-1]]
    )
    change    = scl_stress - scl_base
    direction = "↑ correct   " if change > 0 \
                else "↓ still wrong"

    print(f"S{sid:2d}      "
          f"{scl_base:8.3f}   "
          f"{scl_stress:10.3f}   "
          f"{change:+8.3f}   "
          f"{direction}")

    eda_results.append({
        'sid':    sid,
        'scl_b':  scl_base,
        'scl_s':  scl_stress,
        'change': change
    })

print("-" * 65)

if eda_results:
    correct = sum(1 for r in eda_results if r['change'] > 0)
    n       = len(eda_results)
    avg_b   = np.mean([r['scl_b'] for r in eda_results])
    avg_s   = np.mean([r['scl_s'] for r in eda_results])

    print(f"\nSUMMARY:")
    print(f"  Correct direction: {correct}/{n}")
    print(f"  Mean baseline SCL: {avg_b:.3f} uS")
    print(f"  Mean stress SCL:   {avg_s:.3f} uS")
    print(f"  Mean change:       {avg_s-avg_b:+.3f} uS "
          f"({'↑ correct' if avg_s > avg_b else '↓ wrong'})")

    if correct >= 13:
        print(f"\nEDA FIX VALIDATED")
        print(f"Ready to run full pipeline on all 15 subjects")
    else:
        print(f"\nStill some issues — paste output")

EDA DETREND FIX — VALIDATION ALL SUBJECTS
Expected: stress SCL > baseline SCL for all subjects
Subj   Base SCL   Stress SCL   Change     Direction   
-----------------------------------------------------------------
S 2         1.006        1.574     +0.568   ↑ correct   
S 3         0.865        2.017     +1.152   ↑ correct   
S 4         1.090        1.990     +0.901   ↑ correct   
S 5         2.326        5.338     +3.012   ↑ correct   
S 6         2.043        4.351     +2.307   ↑ correct   
S 7         4.609        7.981     +3.372   ↑ correct   
S 8         0.469        0.612     +0.143   ↑ correct   
S 9         0.571        1.733     +1.162   ↑ correct   
S10         0.745        1.066     +0.321   ↑ correct   
S11         0.555        1.291     +0.736   ↑ correct   
S13         0.898        1.304     +0.405   ↑ correct   
S14         0.454        0.560     +0.106   ↑ correct   
S15         0.480        0.604     +0.124   ↑ correct   
S16         0.494        2.699     +2.205  

In [9]:
# ============================================================
# CELL 8 — FULL PIPELINE ON ALL 15 SUBJECTS
#
# Runs every preprocessing step on all 15 WESAD subjects
# and collects all features into one master list.
#
# Uses:
#   preprocess_ecg()         — Cell 2 validated function
#   preprocess_eda()               — Cell 3 (drift-corrected)
#   process_acc()            — Cell 4 validated function
#   create_pss_labels()      — Cell 5 validated function
#   extract_windows_and_features() — Cell 7 windowing function
#
# Expected output:
#   ~824 windows total
#   ~557 Normal, ~267 Stress
#   Takes 15-30 minutes
# ============================================================

from tqdm import tqdm

print("=" * 65)
print("FULL PREPROCESSING PIPELINE — ALL 15 SUBJECTS")
print("=" * 65)

all_features  = []
all_labels    = []
all_subjects  = []
subject_stats = []

for sid in tqdm(WESAD_SUBJECT_IDS, desc="Processing subjects"):

    print(f"\n{'='*50}")
    print(f"S{sid} processing...")

    # ── Load ─────────────────────────────────────────────────
    pkl_path = os.path.join(
        WESAD_PATH, f"S{sid}", f"S{sid}.pkl"
    )
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    chest      = data['signal']['chest']
    raw_labels = data['label'].flatten()

    # Use raw values directly — already in physical units
    ecg_full = chest['ECG'].flatten().astype(np.float64)
    eda_full = chest['EDA'].flatten().astype(np.float64)
    acc_full = chest['ACC'].astype(np.float64)

    total_min = len(ecg_full) / WESAD_CHEST_FS / 60
    print(f"  Duration: {total_min:.1f} min")

    # ── Step 1: PSS Labels ───────────────────────────────────
    pss_labels = create_pss_labels(
        raw_labels,
        fs=WESAD_CHEST_FS,
        boundary_sec=BOUNDARY_EXCLUSION_SEC
    )
    print(f"  Labels — "
          f"N:{(pss_labels==0).sum()/700/60:.1f}m "
          f"S:{(pss_labels==1).sum()/700/60:.1f}m")

    # ── Step 2: ECG ──────────────────────────────────────────
    print(f"  ECG...", end=" ")
    ibi_full, r_peaks, ibi_times = preprocess_ecg(
        ecg_full, fs=WESAD_CHEST_FS
    )
    if ibi_full is None:
        print(f"FAILED — skipping S{sid}")
        continue
    print(f"done ({len(r_peaks)} peaks)")

    # ── Step 3: EDA (with detrend fix) ───────────────────────
    print(f"  EDA...", end=" ")
    eda_components = preprocess_eda(
        eda_full, fs=WESAD_CHEST_FS
    )
    print(f"done")

    # ── Step 4: ACC ──────────────────────────────────────────
    print(f"  ACC...", end=" ")
    magnitude, clean_mask, motion_pct = process_acc(
        acc_full, fs=WESAD_CHEST_FS
    )
    print(f"done ({motion_pct:.1f}% motion)")

    # ── Step 5: Windowing ────────────────────────────────────
    print(f"  Windowing...", end=" ")
    feats, labs, rejected = extract_windows_and_features(
        ibi_full, ibi_times,
        eda_components,
        magnitude, clean_mask,
        pss_labels,
        fs=WESAD_CHEST_FS,
        window_sec=WINDOW_SEC,
        step_sec=STEP_SEC
    )
    print(f"done — {len(feats)} windows")

    n_n = labs.count(0)
    n_s = labs.count(1)
    print(f"  N:{n_n}  S:{n_s}")
    print(f"  Rejected: {rejected}")

    all_features.extend(feats)
    all_labels.extend(labs)
    all_subjects.extend([sid] * len(feats))

    subject_stats.append({
        'subject':   sid,
        'n_windows': len(feats),
        'n_normal':  n_n,
        'n_stress':  n_s
    })

# ── Final summary ─────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"PIPELINE COMPLETE")
print(f"{'='*65}")
print(f"Total windows extracted: {len(all_features)}")
print(f"  Normal: {all_labels.count(0)}")
print(f"  Stress: {all_labels.count(1)}")
print(f"\nPer subject breakdown:")
for s in subject_stats:
    print(f"  S{s['subject']:2d}: "
          f"total={s['n_windows']:3d}  "
          f"N={s['n_normal']:3d}  "
          f"S={s['n_stress']:3d}")

FULL PREPROCESSING PIPELINE — ALL 15 SUBJECTS


Processing subjects:   0%|          | 0/15 [00:00<?, ?it/s]


S2 processing...
  Duration: 101.3 min
  Labels — N:18.1m S:9.2m
  ECG... done (6898 peaks)
  EDA... done
  ACC... done (0.0% motion)
  Windowing... 

Processing subjects:   7%|▋         | 1/15 [01:26<20:14, 86.74s/it]

done — 59 windows
  N:38  S:21
  Rejected: {'excluded_label': 142, 'mixed_label': 0, 'motion': 0, 'short_ibi': 0, 'feat_error': 0}

S3 processing...
  Duration: 108.2 min
  Labels — N:18.0m S:9.7m
  ECG... done (6408 peaks)
  EDA... done
  ACC... done (0.0% motion)
  Windowing... 

...

done — 62 windows
  N:39  S:23
  Rejected: {'excluded_label': 112, 'mixed_label': 0, 'motion': 0, 'short_ibi': 0, 'feat_error': 0}

S16 processing...
  Duration: 93.8 min
  Labels — N:18.7m S:10.2m
  ECG... done (7591 peaks)
  EDA... done
  ACC... done (0.0% motion)
  Windowing... 

Processing subjects:  93%|█████████▎| 14/15 [17:47<01:14, 74.98s/it]

done — 59 windows
  N:39  S:20
  Rejected: {'excluded_label': 125, 'mixed_label': 0, 'motion': 0, 'short_ibi': 2, 'feat_error': 0}

S17 processing...
  Duration: 98.7 min
  Labels — N:18.7m S:11.1m
  ECG... done (7019 peaks)
  EDA... done
  ACC... done (0.0% motion)
  Windowing... 

Processing subjects: 100%|██████████| 15/15 [19:03<00:00, 76.26s/it]

done — 61 windows
  N:39  S:22
  Rejected: {'excluded_label': 133, 'mixed_label': 0, 'motion': 0, 'short_ibi': 2, 'feat_error': 0}

PIPELINE COMPLETE
Total windows extracted: 913
  Normal: 587
  Stress: 326

Per subject breakdown:
  S 2: total= 59  N= 38  S= 21
  S 3: total= 60  N= 38  S= 22
  S 4: total= 60  N= 39  S= 21
  S 5: total= 61  N= 40  S= 21
  S 6: total= 61  N= 39  S= 22
  S 7: total= 60  N= 39  S= 21
  S 8: total= 62  N= 39  S= 23
  S 9: total= 60  N= 39  S= 21
  S10: total= 63  N= 39  S= 24
  S11: total= 63  N= 40  S= 23
  S13: total= 62  N= 40  S= 22
  S14: total= 60  N= 40  S= 20
  S15: total= 62  N= 39  S= 23
  S16: total= 59  N= 39  S= 20
  S17: total= 61  N= 39  S= 22


In [10]:
# ============================================================
# CELL 9 — CONVERT TO DATAFRAME, VALIDATE AND SAVE
# ============================================================

import pandas as pd

print("Converting to DataFrame...")

features_df               = pd.DataFrame(all_features)
features_df['label']      = all_labels
features_df['subject_id'] = all_subjects

feature_cols = [c for c in features_df.columns
                if c not in ['label', 'subject_id']]

print(f"Shape: {features_df.shape}")
print(f"Feature columns: {len(feature_cols)}")

# ── Handle NaN ───────────────────────────────────────────────
nan_counts   = features_df[feature_cols].isna().sum()
nan_features = nan_counts[nan_counts > 0]

if len(nan_features) > 0:
    print(f"\nNaN features found:")
    for feat, count in nan_features.items():
        pct = count / len(features_df) * 100
        print(f"  {feat:30s}: {count} ({pct:.1f}%)")
    print(f"\nFilling with column median...")
    for col in feature_cols:
        if features_df[col].isna().any():
            features_df[col] = features_df[col].fillna(
                features_df[col].median()
            )
else:
    print(f"No NaN values")

# ── Label distribution ───────────────────────────────────────
print(f"\nFINAL LABEL DISTRIBUTION:")
label_names = {0:'Normal', 1:'Stress'}
for lid in [0, 1]:
    count = (features_df['label'] == lid).sum()
    pct   = count / len(features_df) * 100
    print(f"  Class {lid} ({label_names[lid]:10s}): "
          f"{count:4d} windows ({pct:.1f}%)")

# ── Feature validation by class ──────────────────────────────
print(f"\nFEATURE VALIDATION BY CLASS")
print(f"(Stress should differ from Normal in the expected direction)")
print(f"{'Feature':15s} {'Normal':10s} {'Stress':10s} {'Valid':10s}")
print("-" * 58)

key_feats = [
    ('SDNN',       'drops'),
    ('RMSSD',      'drops'),
    ('MeanHR',     'rises'),
    ('LF_HF',      'rises'),
    ('SCL_mean',   'rises'),
    ('SCL_slope',  'rises'),
    ('SCR_count',  'rises'),
    ('deriv1_mean','rises'),
    ('ACC_mean',   'rises'),
]

for feat, direction in key_feats:
    if feat not in features_df.columns:
        continue
    n_val = features_df[features_df['label']==0][feat].mean()
    s_val = features_df[features_df['label']==1][feat].mean()

    if direction == 'drops':
        valid = "N>S ✓" if n_val > s_val else "✗"
    else:
        valid = "N<S ✓" if n_val < s_val else "✗"

    print(f"{feat:15s} {n_val:8.3f}   "
          f"{s_val:8.3f}   {valid}")

# ── Paper 10 reference check ─────────────────────────────────
print(f"\nPAPER 10 REFERENCE (Normal class mean):")
normal_df = features_df[features_df['label'] == 0]
refs = [
    ('SDNN',    62.3, 'ms'),
    ('RMSSD',   48.6, 'ms'),
    ('MeanHR',  65.0, 'bpm'),
    ('SCL_mean', 4.2, 'uS'),
]
for feat, ref, unit in refs:
    if feat in normal_df.columns:
        val  = normal_df[feat].mean()
        diff = val - ref
        print(f"  {feat:12s}: {val:.2f}{unit}  "
              f"ref={ref}{unit}  diff={diff:+.2f}")

# ── Save ─────────────────────────────────────────────────────
results_dir = os.path.join(RESULTS_PATH, "objective1")
os.makedirs(results_dir, exist_ok=True)
save_path   = os.path.join(results_dir, "wesad_features.csv")

features_df.to_csv(save_path, index=False)
size_mb = os.path.getsize(save_path) / 1024 / 1024

print(f"\nSaved: {save_path}")
print(f"Size:  {size_mb:.1f} MB")
print(f"\nPREPROCESSING COMPLETE")
print(f"wesad_features.csv ready for feature selection")

Converting to DataFrame...
Shape: (913, 42)
Feature columns: 40

NaN features found:
  LF                            : 36 (3.9%)
  HF                            : 6 (0.7%)
  LF_HF                         : 36 (3.9%)
  VLF                           : 913 (100.0%)
  LFnu                          : 36 (3.9%)
  HFnu                          : 6 (0.7%)

Filling with column median...

FINAL LABEL DISTRIBUTION:
  Class 0 (Normal    ):  587 windows (64.3%)
  Class 1 (Stress    ):  326 windows (35.7%)

FEATURE VALIDATION BY CLASS
(Stress should differ from Normal in the expected direction)
Feature         Normal     Stress     Valid     
----------------------------------------------------------
SDNN              67.698     69.542   ✗
RMSSD             50.144     41.653   N>S ✓
MeanHR            72.247     95.001   N<S ✓
LF_HF              2.039      2.658   N<S ✓
SCL_mean           1.168      2.267   N<S ✓
SCL_slope         -0.000      0.000   N<S ✓
SCR_count          3.554      6.755   N<S ✓
